# Z-Score Diagnostic: Null Distribution Generation & Significance Scoring

**Purpose — Supervisor Review**  
This notebook isolates and exposes the two steps that determine how statistical significance is assigned to each network's M-information value:

| Section | What it does | Cells |
|---|---|---|
| **A — Null Distribution** | Generates 100 IID Gaussian surrogate hidden-state matrices, computes M-information for each, producing a null distribution of 
 M values | Cells 4–5 |
| **B — Z-Scoring** | Scores the observed M against that null distribution. Shows the original parametric formula, why it produces values in the thousands, and what the empirical p-value actually says | Cells 7–9 |

**Data source**: Pre-computed sweep results from the parity classification notebook  
(files: `parity_*_zero_m_zscore_sweep.json` in `Project Files/`)

All six parity network checkpoints are copied to `checkpoints/` in this folder.  
No model forward-passes are run here — the notebook uses the already-stored null M values.

In [ ]:
# ─── Standard library ─────────────────────────────────────────────────────────
import json
from pathlib import Path

# ─── Scientific computing ─────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import norm

# ─── Paths ────────────────────────────────────────────────────────────────────
DIAG_DIR     = Path(globals().get('__vsc_ipynb_file__', __file__)).resolve().parent
PROJECT_ROOT = DIAG_DIR.parent.parent          # two levels up from zscore_diagnostic/
SWEEP_DIR    = PROJECT_ROOT / 'Project Files'  # where parity_*_sweep.json files live
CKPT_DIR     = DIAG_DIR / 'checkpoints'        # local copy of parity .pt files

print(f'Diagnostic dir  : {DIAG_DIR}')
print(f'Sweep JSON dir  : {SWEEP_DIR}')
print(f'Checkpoints dir : {CKPT_DIR}')
print(f'Checkpoint files: {[p.name for p in sorted(CKPT_DIR.glob("*.pt"))]}')

In [ ]:
# ─── Load all parity sweep result files ───────────────────────────────────────
# Each JSON file contains the per-subset results for one (network, tensor) pair.
# Every row stores: observed_M_bits, null_M_values (100 draws), M_zscore, M_p_upper.

sweep_files = sorted(SWEEP_DIR.glob('parity_*_zero_m_zscore_sweep.json'))
print(f'Found {len(sweep_files)} sweep files:')
for f in sweep_files:
    print(f'  {f.name}')

# Parse all files into a dict keyed by (network_key, tensor)
all_sweeps = {}
for fpath in sweep_files:
    payload = json.loads(fpath.read_text(encoding='utf-8-sig'))
    results = payload.get('results', [])
    # Derive key from filename: parity_{key}_{tensor}_zero_m_zscore_sweep.json
    stem_parts = fpath.stem.replace('parity_', '').rsplit('_', 4)
    # stem_parts[-1]='sweep', [-2]='zscore', [-3]='m', [-4]='zero', rest=key+tensor
    name_part = fpath.stem.replace('parity_', '').replace('_zero_m_zscore_sweep', '')
    tensor = name_part.rsplit('_', 1)[-1]          # 'mem' or 'spk'
    net_key = name_part.rsplit('_', 1)[0]          # e.g. '2class_local_hom'
    all_sweeps[(net_key, tensor)] = results

print(f'\nLoaded {len(all_sweeps)} (network, tensor) pairs.')
print('Networks:', sorted(set(k for k, _ in all_sweeps)))

---
## ══════════════════════════════════════════════════════════
## SECTION A — NULL DISTRIBUTION GENERATION
## ══════════════════════════════════════════════════════════

### What the code does

For each (network, tensor, subset size) combination the notebook generates a **null distribution** of M-information values by:

1. Drawing `n_null = 100` independent random matrices of shape `(subset_size, n_time_samples)` from **IID Gaussian(0, 1)**  
   → This represents a hidden-state trajectory with **no temporal structure** (zero memory by construction)
2. Computing the lag-1 covariance matrix of each random matrix
3. Passing that covariance to `W_M_calculator` (wimfo library, Gaussian analytical path) to get `M_bits`  
   → Because the data is IID, each draw should give M ≈ 0

The 100 resulting M values form the null distribution.  
The observed M (from the real trained network) is then compared against this.

### Key question for review
> **Is IID Gaussian an appropriate null?**  
> It represents a network with no memory whatsoever. Any positive M in the real network is "above chance" relative to this null. Because IID data truly has M=0, the null M values are extremely small (≈ 1e-6 to 1e-5 bits) and cluster tightly. This matters for the z-score calculation in Section B.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION A — NULL DISTRIBUTION: visualise the stored null M values
# ══════════════════════════════════════════════════════════════════════════════

# Use the 2-class local-homogeneous membrane network as the representative example.
# This is the simplest architecture and its null/observed pattern is typical of all six.
DEMO_NET    = '2class_local_hom'
DEMO_TENSOR = 'mem'
rows = all_sweeps[(DEMO_NET, DEMO_TENSOR)]          # list of per-subset dicts
rows_by_k = {int(r['subset_size']): r for r in rows}

subset_sizes = sorted(rows_by_k)
print(f'Network  : {DEMO_NET}  |  tensor: {DEMO_TENSOR}')
print(f'Subsets  : {subset_sizes}')
print(f'n_null   : {rows[0]["n_null"]} draws per subset')
print()

# ── Print null distribution summary for every subset ──────────────────────────
print(f'{"k":>4}  {"n_samples":>10}  {"null mean":>12}  {"null std":>12}  '
      f'{"obs M":>12}  {"obs / null_mean ratio":>22}')
print('-' * 80)
for k in subset_sizes:
    r    = rows_by_k[k]
    nv   = np.asarray(r['null_M_values'], dtype=np.float64)
    nv   = nv[np.isfinite(nv)]
    obs  = float(r['observed_M_bits'])
    ratio = obs / nv.mean() if nv.mean() > 0 else float('inf')
    print(f'{k:>4}  {int(r["samples"]):>10}  {nv.mean():>12.4e}  '
          f'{nv.std(ddof=1):>12.4e}  {obs:>12.6f}  {ratio:>22.1f}x')

# ── Plot: null distribution histogram for each subset ─────────────────────────
fig, axes = plt.subplots(1, len(subset_sizes), figsize=(16, 3.5), sharey=False)
fig.suptitle(
    f'SECTION A — Null M-Information Distributions  '
    f'(network: {DEMO_NET}, tensor: {DEMO_TENSOR})\n'
    f'Each panel: 100 IID Gaussian null draws. Red line = observed M from trained network.',
    fontsize=10, y=1.03
)

for ax, k in zip(axes, subset_sizes):
    r   = rows_by_k[k]
    nv  = np.asarray(r['null_M_values'], dtype=np.float64)
    nv  = nv[np.isfinite(nv)]
    obs = float(r['observed_M_bits'])

    ax.hist(nv, bins=15, color='steelblue', alpha=0.75, edgecolor='white', linewidth=0.5)
    ax.axvline(obs, color='crimson', linewidth=2.0, label=f'observed\nM={obs:.4f}')
    ax.set_title(f'k={k} neurons', fontsize=9)
    ax.set_xlabel('M (bits)', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2e'))
    ax.legend(fontsize=7, frameon=False)

    # NOTE FOR SUPERVISOR: The observed M (red line) is so far to the right that
    # it doesn't appear on the histogram axis scale — it is orders of magnitude
    # larger than every null draw. We set xlim to null range only to show the
    # null spread; the annotation shows where the observation actually sits.
    ax.set_xlim(nv.min() * 0.5, nv.max() * 1.5)
    ax.annotate(
        f'obs = {obs:.4f}\n(off scale →)',
        xy=(1.0, 0.97), xycoords='axes fraction',
        ha='right', va='top', fontsize=7, color='crimson'
    )

axes[0].set_ylabel('Count (of 100 null draws)', fontsize=8)
plt.tight_layout()
plt.savefig(DIAG_DIR / 'diag_A_null_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: diag_A_null_distributions.png')

---
## ══════════════════════════════════════════════════════════
## SECTION B — Z-SCORE COMPUTATION
## ══════════════════════════════════════════════════════════

### Two approaches are shown and compared

#### Approach 1 — Parametric z  (original formula, currently in both notebooks)
$$z_{\text{param}} = \frac{M_{\text{obs}} - \mu_{\text{null}}}{\sigma_{\text{null}}}$$

- Mathematically correct, but produces values in the **thousands**  
- Root cause: $\sigma_{\text{null}} \approx 10^{-5}$ because IID Gaussian data yields near-zero M with very small spread
- A z of 16,000 and a z of 5,000 are **statistically indistinguishable** — both mean p ≈ 0
- Not comparable to conventional z-score thresholds (e.g. z > 1.96 for p < 0.05)

#### Approach 2 — Empirical p-value  (recommended primary statistic)
$$p_{\text{upper}} = \frac{1 + \#\{\text{null} \geq M_{\text{obs}}\}}{n_{\text{null}} + 1}$$

- With $n=100$ draws, the finest achievable p is $\frac{1}{101} \approx 0.0099$
- p = 0.0099 means **every single null draw** was below the observed M
- This is an **honest, distribution-free** significance statement

#### Approach 3 — norm.ppf z-equivalent  (proposed fix)
$$z_{\text{ppf}} = \Phi^{-1}(1 - p_{\text{upper}})$$

- Converts empirical p to a normal-quantile z  
- **Cap with n=100**: max z = $\Phi^{-1}(100/101) \approx 2.33$  
- **Problem for this dataset**: all networks beat all 100 null draws at every subset size → all z = 2.33 exactly, destroying ranking information

### Key question for review
> **Which approach should be used for the analysis and report?**  
> The empirical p-value (Approach 2) is the most statistically honest. The parametric z (Approach 1) is correct in formula but numerically extreme. The norm.ppf z (Approach 3) is bounded but flattens all results to the same value with n=100.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION B — Z-SCORE COMPUTATION: compare all three approaches for the demo network
# ══════════════════════════════════════════════════════════════════════════════

def zscore_parametric(observed, null_values):
    """APPROACH 1 — Original formula: (obs - null_mean) / null_std.

    Mathematically correct, but produces extreme values when null_std is near-zero.
    This is the formula currently used in both parity and sample-subset notebooks.
    """
    valid = np.asarray(null_values, dtype=np.float64)
    valid = valid[np.isfinite(valid)]
    if valid.size == 0:
        return np.nan, np.nan
    mu  = float(valid.mean())
    std = float(valid.std(ddof=1)) if valid.size > 1 else np.nan
    # ── This line is where the explosion happens ───────────────────────────────
    # std ≈ 1e-5 → z = (0.95 - 0.0005) / 0.00005 ≈ 16,000
    z = float((observed - mu) / std) if (np.isfinite(std) and std > 1e-12) else np.nan
    return z, mu


def zscore_empirical_p(observed, null_values):
    """APPROACH 2 — Empirical upper-tail p-value.

    No distributional assumption. p = (1 + #{null >= obs}) / (n + 1).
    With n=100, finest achievable p = 1/101 ≈ 0.0099 (observation beats all nulls).
    """
    valid = np.asarray(null_values, dtype=np.float64)
    valid = valid[np.isfinite(valid)]
    if valid.size == 0:
        return np.nan
    p = (1.0 + np.sum(valid >= float(observed))) / (valid.size + 1.0)
    return float(p)


def zscore_norm_ppf(observed, null_values):
    """APPROACH 3 — Empirical p converted to a normal-quantile z via norm.ppf.

    Bounded by n: with n=100, max|z| = norm.ppf(100/101) ≈ 2.33.
    Problem: when all networks beat all null draws, every z collapses to 2.33.
    """
    p = zscore_empirical_p(observed, null_values)
    if not np.isfinite(p):
        return np.nan
    p_clamped = float(np.clip(p, 1e-9, 1.0 - 1e-9))
    return float(norm.ppf(1.0 - p_clamped))


# ── Compute all three for the demo network across all subset sizes ─────────────
print(f'Network: {DEMO_NET}  |  tensor: {DEMO_TENSOR}  |  n_null = 100')
print()
header = (f'{"k":>4}  {"obs_M":>10}  {"null_mean":>12}  {"null_std":>12}  '
          f'{"z_param":>10}  {"p_upper":>8}  {"z_ppf":>7}  {"verdict"}')
print(header)
print('-' * len(header))

results_demo = []
for k in subset_sizes:
    r    = rows_by_k[k]
    obs  = float(r['observed_M_bits'])
    nv   = np.asarray(r['null_M_values'], dtype=np.float64)
    nv   = nv[np.isfinite(nv)]

    z_p, mu = zscore_parametric(obs, nv)
    p_e     = zscore_empirical_p(obs, nv)
    z_n     = zscore_norm_ppf(obs, nv)

    # NOTE FOR SUPERVISOR: The verdict is based on the empirical p-value, which
    # is the most reliable statistic here given the near-zero null variance.
    verdict = '*** ALL 100 nulls below obs ***' if p_e <= 1 / (len(nv) + 0.5) else f'p={p_e:.4f}'

    print(f'{k:>4}  {obs:>10.6f}  {mu:>12.4e}  {nv.std(ddof=1):>12.4e}  '
          f'{z_p:>10.1f}  {p_e:>8.4f}  {z_n:>7.3f}  {verdict}')

    results_demo.append(dict(k=k, obs=obs, mu=mu, std=nv.std(ddof=1),
                             z_param=z_p, p=p_e, z_ppf=z_n))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION B (continued) — Cross-network comparison
# Show that the z_ppf (norm.ppf) approach collapses ALL networks to the same
# value (2.33) because every network beats every null draw.
# This demonstrates why the fix loses discriminability.
# ══════════════════════════════════════════════════════════════════════════════

# Pick subset k=16 as the representative comparison point.
DEMO_K = 16
print(f'Cross-network comparison at k={DEMO_K} neurons, tensor=mem')
print()
print(f'{"network":>32}  {"obs_M":>8}  {"z_param":>10}  {"p_upper":>8}  {"z_ppf":>7}')
print('-' * 72)

all_net_keys = sorted(set(k for k, t in all_sweeps if t == 'mem'))
for net_key in all_net_keys:
    rows_net = all_sweeps.get((net_key, 'mem'), [])
    row = next((r for r in rows_net if int(r['subset_size']) == DEMO_K), None)
    if row is None:
        print(f'{net_key:>32}  (missing k={DEMO_K})')
        continue
    obs = float(row['observed_M_bits'])
    nv  = np.asarray(row['null_M_values'], dtype=np.float64)
    nv  = nv[np.isfinite(nv)]
    z_p, _ = zscore_parametric(obs, nv)
    p_e    = zscore_empirical_p(obs, nv)
    z_n    = zscore_norm_ppf(obs, nv)
    print(f'{net_key:>32}  {obs:>8.4f}  {z_p:>10.1f}  {p_e:>8.4f}  {z_n:>7.3f}')

print()
print('NOTE FOR SUPERVISOR:')
print('  z_ppf is identical for all networks (2.326) because every network beats')
print('  all 100 null draws — p = 1/101 = 0.0099 in every case.')
print('  The parametric z_param differs between networks, preserving rank order,')
print('  but is numerically uninterpretable as a conventional z-score.')
print('  The empirical p-value (p=0.0099) is the most honest and meaningful statistic.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION B (continued) — Diagnostic comparison plot
# Three panels showing the three approaches side-by-side for the demo network.
# ══════════════════════════════════════════════════════════════════════════════

ks     = [d['k']      for d in results_demo]
z_p    = [d['z_param'] for d in results_demo]
p_vals = [d['p']       for d in results_demo]
z_n    = [d['z_ppf']   for d in results_demo]
neg_log_p = [-np.log10(p) for p in p_vals]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    f'SECTION B — Z-Score Approaches Compared\n'
    f'Network: {DEMO_NET}, tensor: {DEMO_TENSOR}',
    fontsize=11
)

# ── Panel 1: Parametric z (explodes into thousands) ────────────────────────────
axes[0].bar(range(len(ks)), z_p, color='salmon', edgecolor='darkred', linewidth=0.8)
axes[0].set_xticks(range(len(ks)))
axes[0].set_xticklabels([f'k={k}' for k in ks])
axes[0].set_title('Approach 1: Parametric z\n(obs - null_mean) / null_std', fontsize=9)
axes[0].set_ylabel('z (parametric)', fontsize=9)
axes[0].set_xlabel('Subset size (neurons)', fontsize=9)
for i, v in enumerate(z_p):
    axes[0].text(i, v * 0.9, f'{v:.0f}', ha='center', va='top', fontsize=8, color='darkred')
# NOTE: these are correct mathematically but uninterpretable — z=16,959 and z=9,443
# both simply mean "far above null"; the difference is not meaningful.

# ── Panel 2: –log10(p) — the honest significance metric ───────────────────────
axes[1].bar(range(len(ks)), neg_log_p, color='steelblue', edgecolor='navy', linewidth=0.8)
axes[1].axhline(2.0, color='orange', linestyle='--', linewidth=1.2,
                label='p = 0.01 threshold\n(n=100 resolution limit)')
axes[1].set_xticks(range(len(ks)))
axes[1].set_xticklabels([f'k={k}' for k in ks])
axes[1].set_title('Approach 2: −log₁₀(p_empirical)\nRecommended significance metric', fontsize=9)
axes[1].set_ylabel('−log₁₀(p)', fontsize=9)
axes[1].set_xlabel('Subset size (neurons)', fontsize=9)
axes[1].legend(fontsize=7, frameon=False)
# NOTE: all values hit the ceiling of −log10(1/101) ≈ 2.0 because every
# observation beats all 100 null draws. This IS the resolution limit — not a bug.

# ── Panel 3: norm.ppf z (proposed fix — collapses to 2.33) ────────────────────
axes[2].bar(range(len(ks)), z_n, color='mediumseagreen', edgecolor='darkgreen', linewidth=0.8)
axes[2].axhline(1.96, color='orange', linestyle='--', linewidth=1.2, label='z=1.96 (p=0.05)')
axes[2].set_xticks(range(len(ks)))
axes[2].set_xticklabels([f'k={k}' for k in ks])
axes[2].set_title('Approach 3: norm.ppf(1 − p)\n(Bounded z — max=2.33 with n=100)', fontsize=9)
axes[2].set_ylabel('z (norm.ppf)', fontsize=9)
axes[2].set_xlabel('Subset size (neurons)', fontsize=9)
axes[2].set_ylim(0, 3.0)
axes[2].legend(fontsize=7, frameon=False)
# NOTE: every bar is identical (2.326) because p=0.0099 for all subsets.
# This approach loses all ranking information with n=100 draws.

plt.tight_layout()
plt.savefig(DIAG_DIR / 'diag_B_zscore_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: diag_B_zscore_comparison.png')

print()
print('═' * 70)
print('SUMMARY FOR SUPERVISOR')
print('═' * 70)
print('  • The null M values are genuinely near-zero (IID Gaussian → M ≈ 0).')
print('  • The observed M values are orders of magnitude larger (real network')
print('    has genuine temporal structure / memory).')
print('  • The parametric z formula is correct but produces extreme values')
print('    because null_std ≈ 1e-5.')
print('  • The empirical p = 0.0099 is the most meaningful statistic:')
print('    every observed M beat every null draw.')
print('  • The norm.ppf fix is valid in principle but destroys ranking with n=100.')
print('  • Recommendation: report empirical p and –log10(p) as the primary')
print('    significance metrics. Increase n_null to 1000+ for z-score resolution.')
print('═' * 70)

---
## SECTION D — Independent TDMI-Matched Null Scoring (Homogeneous + Log-Uniform)

This section is **independent** from earlier sections and reruns null scoring from scratch for only:

- `2class_local_hom`
- `2class_fittedhet_lu`
- `4class_local_hom`
- `4class_fittedhet_lu`

for both `mem` and `spk` tensors across subset sizes.

### Method (independent rerun)

1. Load observed `M` and `W` from the existing parity sweep JSON files.
2. Compute observed TDMI as:
   $$\mathrm{TDMI}_{\mathrm{obs}} = W_{\mathrm{obs}} + M_{\mathrm{obs}}$$
3. Build null systems using random stable VAR(1) dynamics, and tune spectral radius so each null draw matches `TDMI_obs`.
4. Compute null `M` values with `W_M_calculator(..., option="distr", type="gaussian")`.
5. Score observed `M` against TDMI-matched null via:
   - Parametric z: $(M_{obs}-\mu_{null})/\sigma_{null}$
   - Empirical upper-tail p-value: $p=(1+\#\{M_{null}\ge M_{obs}\})/(n_{null}+1)$

The outputs below provide the independent z-scores, p-values, and visual diagnostics.

In [1]:
# SECTION D setup (independent)
import io
import json
import sys
from contextlib import redirect_stdout
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.linalg import solve_discrete_lyapunov

# Resolve notebook/project paths
try:
    DIAG_DIR_D = Path(globals()['__vsc_ipynb_file__']).resolve().parent
except Exception:
    DIAG_DIR_D = Path(r"C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\zscore_diagnostic")

PROJECT_ROOT_D = DIAG_DIR_D.parent.parent
SWEEP_DIR_D = PROJECT_ROOT_D / "Project Files"
WIMFO_ROOT_D = PROJECT_ROOT_D / "wimfo"

if str(WIMFO_ROOT_D) not in sys.path:
    sys.path.insert(0, str(WIMFO_ROOT_D))

from wimfo.W_M_Info import W_M_calculator

TARGET_NETS_D = [
    "2class_local_hom",
    "2class_fittedhet_lu",
    "4class_local_hom",
    "4class_fittedhet_lu",
]
TARGET_TENSORS_D = ["mem", "spk"]
N_NULL_D = 40  # increase if you want finer p-value resolution
RIDGE_D = 1e-2

# Load only target sweeps
all_sweeps_d = {}
for net in TARGET_NETS_D:
    for ten in TARGET_TENSORS_D:
        fpath = SWEEP_DIR_D / f"parity_{net}_{ten}_zero_m_zscore_sweep.json"
        if not fpath.exists():
            print(f"[missing] {fpath.name}")
            continue
        payload = json.loads(fpath.read_text(encoding="utf-8-sig"))
        all_sweeps_d[(net, ten)] = payload.get("results", [])

print(f"Loaded {len(all_sweeps_d)} target sweeps.")
print(f"n_null (TDMI-matched) = {N_NULL_D}")
print(f"Sweep dir: {SWEEP_DIR_D}")

Loaded 8 target sweeps.
n_null (TDMI-matched) = 40
Sweep dir: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files


In [2]:
# SECTION D helpers: TDMI-matched null generator + scoring utilities

def _regularize_cov(cov, ridge=1e-2):
    cov = np.asarray(cov, dtype=np.float64)
    cov = np.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0)
    cov = 0.5 * (cov + cov.T)
    scale = np.trace(cov) / max(cov.shape[0], 1)
    if (not np.isfinite(scale)) or scale <= 0.0:
        scale = 1.0
    return cov + float(ridge) * scale * np.eye(cov.shape[0], dtype=np.float64)


def _wm_from_lagged_cov(lagged_cov, ridge=1e-2):
    lagged_cov = _regularize_cov(lagged_cov, ridge=ridge)
    with io.StringIO() as buf, redirect_stdout(buf):
        w_bits, m_bits = W_M_calculator(
            lagged_cov,
            option="distr",
            type="gaussian",
            unit="bits",
            verbose=False,
            optimiser="Newton",
            options=None,
        )
    return float(w_bits), float(m_bits)


def _var1_tmi_bits(A, eps_cov):
    """Return TDMI(X_t; X_{t+1}) in bits for stationary VAR(1): x_{t+1}=A x_t+eps."""
    S = solve_discrete_lyapunov(A, eps_cov)  # stationary covariance
    S = 0.5 * (S + S.T)
    Cpf = S @ A.T  # Cov(past, future)
    block = np.block([[S, Cpf], [Cpf.T, S]])
    s1, l1 = np.linalg.slogdet(S)
    sb, lb = np.linalg.slogdet(block)
    if s1 <= 0 or sb <= 0:
        return np.nan
    return 0.5 * ((l1 + l1) - lb) / np.log(2)


def _build_tdmi_matched_lagged_cov(nvar, tmi_target_bits, rng, g_max=0.9995, tol=1e-3):
    """Sample random VAR(1), tune spectral radius to match target TDMI, return lagged covariance."""
    n = int(nvar)
    eps_cov = np.eye(n, dtype=np.float64)

    # random A template with unit spectral radius
    A0 = rng.normal(0.0, 1.0, size=(n, n))
    eig = np.linalg.eigvals(A0)
    sr = float(np.max(np.abs(eig)))
    if (not np.isfinite(sr)) or sr <= 1e-12:
        return None
    A_unit = A0 / sr

    # bracket TDMI(g)
    lo, hi = 1e-6, float(g_max)
    try:
        t_lo = _var1_tmi_bits(lo * A_unit, eps_cov)
        t_hi = _var1_tmi_bits(hi * A_unit, eps_cov)
    except Exception:
        return None

    if (not np.isfinite(t_lo)) or (not np.isfinite(t_hi)):
        return None

    # if the sampled system cannot reach target TDMI under stability cap, reject draw
    if t_hi < float(tmi_target_bits):
        return None

    # binary search g s.t. TDMI(g) ~= target
    for _ in range(64):
        mid = 0.5 * (lo + hi)
        try:
            t_mid = _var1_tmi_bits(mid * A_unit, eps_cov)
        except Exception:
            return None
        if not np.isfinite(t_mid):
            return None
        if abs(t_mid - float(tmi_target_bits)) <= float(tol):
            lo = hi = mid
            break
        if t_mid < float(tmi_target_bits):
            lo = mid
        else:
            hi = mid

    g = 0.5 * (lo + hi)
    A = g * A_unit

    # build lagged covariance block [past;future]
    try:
        S = solve_discrete_lyapunov(A, eps_cov)
    except Exception:
        return None
    S = 0.5 * (S + S.T)
    Cpf = S @ A.T
    lagged_cov = np.block([[S, Cpf], [Cpf.T, S]])
    return lagged_cov


def build_tdmi_null_distribution(nvar, tmi_target_bits, n_null=40, seed=12345, ridge=1e-2):
    rng = np.random.default_rng(int(seed))
    null_m = []
    null_w = []
    fail_count = 0

    while len(null_m) < int(n_null):
        lag_cov = _build_tdmi_matched_lagged_cov(nvar, tmi_target_bits, rng)
        if lag_cov is None:
            fail_count += 1
            if fail_count > 500:
                break
            continue
        try:
            w_bits, m_bits = _wm_from_lagged_cov(lag_cov, ridge=ridge)
            if np.isfinite(w_bits) and np.isfinite(m_bits):
                null_w.append(float(w_bits))
                null_m.append(float(m_bits))
            else:
                fail_count += 1
        except Exception:
            fail_count += 1

    return {
        "null_M_values": null_m,
        "null_W_values": null_w,
        "n_null_valid": int(len(null_m)),
        "n_fail": int(fail_count),
    }


def score_observed_vs_null_m(observed_m, null_m_values):
    vals = np.asarray(null_m_values, dtype=np.float64)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return {"z": np.nan, "p_upper": np.nan, "null_mean": np.nan, "null_std": np.nan}
    mu = float(vals.mean())
    sd = float(vals.std(ddof=1)) if vals.size > 1 else np.nan
    z = float((float(observed_m) - mu) / sd) if (np.isfinite(sd) and sd > 1e-12) else np.nan
    p = float((1.0 + np.sum(vals >= float(observed_m))) / (vals.size + 1.0))
    return {"z": z, "p_upper": p, "null_mean": mu, "null_std": sd}

In [3]:
# SECTION D run: independent TDMI-null scoring for hom + log-uniform networks

records_d = []

for (net_key, tensor), rows in sorted(all_sweeps_d.items(), key=lambda x: (x[0][0], x[0][1])):
    print(f"\nScoring {net_key} / {tensor}")
    for row in sorted(rows, key=lambda r: int(r.get("subset_size", 0))):
        k = int(row["subset_size"])
        obs_m = float(row["observed_M_bits"])
        obs_w = float(row["observed_W_bits"])
        tdmi_obs = obs_w + obs_m

        # deterministic seed per (network, tensor, k)
        base_seed = abs(hash((net_key, tensor, k))) % (2**31 - 1)

        null_dist = build_tdmi_null_distribution(
            nvar=k,
            tmi_target_bits=tdmi_obs,
            n_null=N_NULL_D,
            seed=base_seed,
            ridge=RIDGE_D,
        )

        scored = score_observed_vs_null_m(obs_m, null_dist["null_M_values"])

        records_d.append({
            "network": net_key,
            "tensor": tensor,
            "subset_size": k,
            "observed_M_bits": obs_m,
            "observed_W_bits": obs_w,
            "observed_TDMI_bits": tdmi_obs,
            "null_M_mean": scored["null_mean"],
            "null_M_std": scored["null_std"],
            "M_z_tdmi_null": scored["z"],
            "M_p_upper_tdmi_null": scored["p_upper"],
            "n_null_valid": int(null_dist["n_null_valid"]),
            "n_null_fail": int(null_dist["n_fail"]),
            "null_M_values_tdmi": null_dist["null_M_values"],
        })

        print(
            f"  k={k:2d} | obs_M={obs_m:8.5f} | TDMI_obs={tdmi_obs:8.4f} | "
            f"z={scored['z']:8.3f} | p={scored['p_upper']:.4f} | n={null_dist['n_null_valid']:2d}"
        )

scores_tdmi_df = pd.DataFrame.from_records(records_d)

# Keep a compact view for notebook display
show_cols = [
    "network", "tensor", "subset_size", "observed_M_bits", "observed_TDMI_bits",
    "null_M_mean", "null_M_std", "M_z_tdmi_null", "M_p_upper_tdmi_null", "n_null_valid"
]
scores_tdmi_df = scores_tdmi_df.sort_values(["network", "tensor", "subset_size"]).reset_index(drop=True)

print("\nIndependent TDMI-null scoring complete.")
print(scores_tdmi_df[show_cols].to_string(index=False))

# Save full results for reproducibility
csv_path = DIAG_DIR_D / "sectionD_tdmi_null_scores.csv"
scores_tdmi_df.drop(columns=["null_M_values_tdmi"]).to_csv(csv_path, index=False)
print(f"\nSaved summary CSV: {csv_path}")


Scoring 2class_fittedhet_lu / mem
  k= 2 | obs_M=-0.00169 | TDMI_obs=  2.5113 | z=  -1.183 | p=1.0000 | n=40
  k= 4 | obs_M= 0.12208 | TDMI_obs=  6.7617 | z=  -1.408 | p=1.0000 | n=40
  k= 8 | obs_M= 0.40500 | TDMI_obs=  8.0915 | z=  -2.243 | p=1.0000 | n=40
  k=16 | obs_M= 1.65116 | TDMI_obs= 13.7093 | z=  -3.018 | p=0.9756 | n=40
  k=32 | obs_M= 3.55096 | TDMI_obs= 15.2984 | z=  -7.405 | p=1.0000 | n=40

Scoring 2class_fittedhet_lu / spk
  k= 2 | obs_M= 0.04413 | TDMI_obs=  4.9280 | z=  -0.852 | p=0.7561 | n=40
  k= 4 | obs_M= 0.19338 | TDMI_obs=  7.7703 | z=  -1.037 | p=0.9512 | n=40
  k= 8 | obs_M= 0.49648 | TDMI_obs=  6.7984 | z=  -2.338 | p=1.0000 | n=40
  k=16 | obs_M= 1.50826 | TDMI_obs=  4.8863 | z=  -0.216 | p=0.5854 | n=40
  k=32 | obs_M= 4.29417 | TDMI_obs= 13.9796 | z=  -0.599 | p=0.7073 | n=40

Scoring 2class_local_hom / mem
  k= 2 | obs_M= 0.00243 | TDMI_obs=  5.0196 | z=  -0.547 | p=0.9512 | n=40
  k= 4 | obs_M= 0.04798 | TDMI_obs=  8.9358 | z=  -1.135 | p=1.0000 | n=4

KeyboardInterrupt: 

In [ ]:
# SECTION D visuals

plot_df = scores_tdmi_df.copy()
plot_df["network_short"] = (
    plot_df["network"]
    .str.replace("2class_", "2c_", regex=False)
    .str.replace("4class_", "4c_", regex=False)
    .str.replace("fittedhet_lu", "loguniform", regex=False)
    .str.replace("local_hom", "homogeneous", regex=False)
)

# 1) Heatmaps of z-scores (one per tensor)
for tensor in ["mem", "spk"]:
    sub = plot_df[plot_df["tensor"] == tensor]
    pivot = sub.pivot(index="network_short", columns="subset_size", values="M_z_tdmi_null")
    pivot = pivot.reindex(sorted(pivot.index))
    fig, ax = plt.subplots(figsize=(8, 3.8))
    mat = pivot.values.astype(float)
    vmax = np.nanmax(np.abs(mat)) if np.isfinite(mat).any() else 1.0
    vmax = max(vmax, 1.0)
    im = ax.imshow(mat, cmap="coolwarm", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_title(f"SECTION D: TDMI-null M z-scores ({tensor})")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"k={int(c)}" for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(list(pivot.index))
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("z-score")
    plt.tight_layout()
    outp = DIAG_DIR_D / f"sectionD_tdmi_z_heatmap_{tensor}.png"
    plt.savefig(outp, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {outp}")

# 2) -log10(p) lines by subset size
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, tensor in zip(axes, ["mem", "spk"]):
    sub = plot_df[plot_df["tensor"] == tensor].copy()
    for net in sorted(sub["network_short"].unique()):
        s = sub[sub["network_short"] == net].sort_values("subset_size")
        y = -np.log10(np.clip(s["M_p_upper_tdmi_null"].values.astype(float), 1e-12, 1.0))
        ax.plot(s["subset_size"], y, marker="o", label=net)
    ax.set_title(f"SECTION D: TDMI-null significance ({tensor})")
    ax.set_xlabel("Subset size")
    ax.set_ylabel("-log10(p_upper)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
outp = DIAG_DIR_D / "sectionD_tdmi_pvalue_lines.png"
plt.savefig(outp, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {outp}")

# 3) Example null histograms at k=16 for membrane values
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
mem16 = plot_df[(plot_df["tensor"] == "mem") & (plot_df["subset_size"] == 16)]
for ax, (_, r) in zip(axes.flatten(), mem16.iterrows()):
    null_vals = np.asarray(r["null_M_values_tdmi"], dtype=np.float64)
    null_vals = null_vals[np.isfinite(null_vals)]
    ax.hist(null_vals, bins=12, color="steelblue", alpha=0.8, edgecolor="white")
    ax.axvline(float(r["observed_M_bits"]), color="crimson", lw=2)
    ax.set_title(f"{r['network_short']} mem k=16\nz={r['M_z_tdmi_null']:.2f}, p={r['M_p_upper_tdmi_null']:.4f}")
    ax.set_xlabel("M (bits)")
    ax.set_ylabel("count")
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2e"))
plt.tight_layout()
outp = DIAG_DIR_D / "sectionD_tdmi_hist_mem_k16.png"
plt.savefig(outp, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {outp}")

In [4]:
# SECTION D AUDIT — TDMI match quality and potential bias checks
# This cell audits whether null generation is truly matching the intended TDMI,
# and whether post-regularization / solver steps drift away from the target.

import hashlib
from wimfo.utils.utils_gauss import tdmi_from_cov

def _stable_seed(net_key, tensor, k):
    token = f"{net_key}|{tensor}|{int(k)}".encode("utf-8")
    # deterministic across sessions (unlike Python's built-in hash)
    return int(hashlib.sha256(token).hexdigest()[:8], 16)


def _get_observed_row(net_key, tensor, k):
    rows = all_sweeps_d[(net_key, tensor)]
    for r in rows:
        if int(r["subset_size"]) == int(k):
            return r
    return None


audit_cases = [
    ("2class_fittedhet_lu", "spk", 8),
    ("2class_fittedhet_lu", "spk", 16),
    ("2class_local_hom", "spk", 8),
]

n_draws_audit = 6
print(f"Audit draws per case: {n_draws_audit}")
print()

for net_key, tensor, k in audit_cases:
    obs_row = _get_observed_row(net_key, tensor, k)
    if obs_row is None:
        print(f"[missing] {net_key}/{tensor}/k={k}")
        continue

    obs_m = float(obs_row["observed_M_bits"])
    obs_w = float(obs_row["observed_W_bits"])
    tmi_target = obs_w + obs_m

    rng = np.random.default_rng(_stable_seed(net_key, tensor, k))

    tdmi_raw_vals = []
    tdmi_reg_vals = []
    tdmi_wm_vals = []
    m_vals = []
    n_none = 0
    n_fail_wm = 0

    for _ in range(n_draws_audit):
        lag_cov = _build_tdmi_matched_lagged_cov(k, tmi_target, rng)
        if lag_cov is None:
            n_none += 1
            continue

        # TDMI at generation stage (what binary search is targeting)
        try:
            td_raw = float(tdmi_from_cov(lag_cov, xdim=int(k)))
        except Exception:
            td_raw = np.nan

        # TDMI after regularization (what actually goes into W_M_calculator)
        lag_cov_reg = _regularize_cov(lag_cov, ridge=RIDGE_D)
        try:
            td_reg = float(tdmi_from_cov(lag_cov_reg, xdim=int(k)))
        except Exception:
            td_reg = np.nan

        try:
            w_bits, m_bits = _wm_from_lagged_cov(lag_cov, ridge=RIDGE_D)
            td_wm = w_bits + m_bits
            m_vals.append(float(m_bits))
            tdmi_wm_vals.append(float(td_wm))
        except Exception:
            n_fail_wm += 1
            w_bits = np.nan
            m_bits = np.nan
            td_wm = np.nan

        tdmi_raw_vals.append(td_raw)
        tdmi_reg_vals.append(td_reg)

    tdmi_raw_vals = np.asarray(tdmi_raw_vals, dtype=np.float64)
    tdmi_reg_vals = np.asarray(tdmi_reg_vals, dtype=np.float64)
    tdmi_wm_vals = np.asarray(tdmi_wm_vals, dtype=np.float64)
    m_vals = np.asarray(m_vals, dtype=np.float64)

    mu_m = float(np.nanmean(m_vals)) if m_vals.size else np.nan
    sd_m = float(np.nanstd(m_vals, ddof=1)) if m_vals.size > 1 else np.nan
    z_val = float((obs_m - mu_m) / sd_m) if (np.isfinite(sd_m) and sd_m > 1e-12) else np.nan

    print(f"CASE: {net_key} / {tensor} / k={k}")
    print(f"  observed M={obs_m:.6f}, observed W={obs_w:.6f}, target TDMI={tmi_target:.6f}")
    print(f"  generated draws: valid={len(tdmi_raw_vals)} none={n_none} wm_fail={n_fail_wm}")

    if len(tdmi_raw_vals):
        print(
            f"  TDMI raw (binary-search target): mean={np.nanmean(tdmi_raw_vals):.6f}, "
            f"sd={np.nanstd(tdmi_raw_vals):.6f}, rel.err={(np.nanmean(tdmi_raw_vals)-tmi_target)/tmi_target:+.2%}"
        )
    if len(tdmi_reg_vals):
        print(
            f"  TDMI after regularization      : mean={np.nanmean(tdmi_reg_vals):.6f}, "
            f"sd={np.nanstd(tdmi_reg_vals):.6f}, rel.err={(np.nanmean(tdmi_reg_vals)-tmi_target)/tmi_target:+.2%}"
        )
    if len(tdmi_wm_vals):
        print(
            f"  TDMI via W+M from solver       : mean={np.nanmean(tdmi_wm_vals):.6f}, "
            f"sd={np.nanstd(tdmi_wm_vals):.6f}, rel.err={(np.nanmean(tdmi_wm_vals)-tmi_target)/tmi_target:+.2%}"
        )

    print(
        f"  null M mean={mu_m:.6f}, null M sd={sd_m:.6f}, "
        f"observed-vs-null z={z_val:.3f}"
    )
    print("-")

Audit draws per case: 6

CASE: 2class_fittedhet_lu / spk / k=8
  observed M=0.496480, observed W=6.301953, target TDMI=6.798433
  generated draws: valid=6 none=0 wm_fail=0
  TDMI raw (binary-search target): mean=6.798290, sd=0.000355, rel.err=-0.00%
  TDMI after regularization      : mean=5.907519, sd=0.676305, rel.err=-13.10%
  TDMI via W+M from solver       : mean=5.907519, sd=0.676305, rel.err=-13.10%
  null M mean=1.615286, null M sd=0.666662, observed-vs-null z=-1.678
-
CASE: 2class_fittedhet_lu / spk / k=16
  observed M=1.508258, observed W=3.378025, target TDMI=4.886283
  generated draws: valid=6 none=0 wm_fail=0
  TDMI raw (binary-search target): mean=4.886274, sd=0.000594, rel.err=-0.00%
  TDMI after regularization      : mean=4.755303, sd=0.001895, rel.err=-2.68%
  TDMI via W+M from solver       : mean=4.755303, sd=0.001895, rel.err=-2.68%
  null M mean=1.545941, null M sd=0.064956, observed-vs-null z=-0.580
-
CASE: 2class_local_hom / spk / k=8
  observed M=0.062073, observed

In [5]:
# SECTION D AUDIT 2 — sensitivity to ridge regularization for one representative case
from wimfo.utils.utils_gauss import tdmi_from_cov

net_key, tensor, k = "2class_fittedhet_lu", "spk", 8
obs_row = None
for r in all_sweeps_d[(net_key, tensor)]:
    if int(r["subset_size"]) == int(k):
        obs_row = r
        break

obs_m = float(obs_row["observed_M_bits"])
obs_w = float(obs_row["observed_W_bits"])
target = obs_m + obs_w
rng = np.random.default_rng(20260601)

print(f"Case: {net_key}/{tensor}/k={k}")
print(f"Observed M={obs_m:.6f}, W={obs_w:.6f}, TDMI target={target:.6f}")
print()

for ridge in [0.0, 1e-4, 1e-3, 1e-2]:
    m_vals = []
    td_vals = []
    n_ok = 0
    n_none = 0

    for _ in range(6):
        lag_cov = _build_tdmi_matched_lagged_cov(k, target, rng)
        if lag_cov is None:
            n_none += 1
            continue
        try:
            # apply chosen ridge exactly as in scoring path
            cov = _regularize_cov(lag_cov, ridge=ridge)
            w, m = _wm_from_lagged_cov(lag_cov, ridge=ridge)
            m_vals.append(float(m))
            td_vals.append(float(w + m))
            n_ok += 1
        except Exception:
            pass

    m_vals = np.asarray(m_vals, dtype=np.float64)
    td_vals = np.asarray(td_vals, dtype=np.float64)

    mu = float(np.nanmean(m_vals)) if n_ok else np.nan
    sd = float(np.nanstd(m_vals, ddof=1)) if n_ok > 1 else np.nan
    z = float((obs_m - mu) / sd) if (np.isfinite(sd) and sd > 1e-12) else np.nan

    td_mu = float(np.nanmean(td_vals)) if n_ok else np.nan
    rel_err = (td_mu - target) / target if (np.isfinite(td_mu) and target > 0) else np.nan

    print(
        f"ridge={ridge:>7.0e} | n_ok={n_ok} n_none={n_none} | "
        f"TDMI(W+M) mean={td_mu:8.4f} ({rel_err:+6.1%}) | "
        f"null M mean={mu:7.4f}, sd={sd:7.4f}, z={z:7.3f}"
    )

Case: 2class_fittedhet_lu/spk/k=8
Observed M=0.496480, W=6.301953, TDMI target=6.798433

ridge=  0e+00 | n_ok=6 n_none=0 | TDMI(W+M) mean=  6.7980 ( -0.0%) | null M mean= 1.8638, sd= 0.3358, z= -4.071
ridge=  1e-04 | n_ok=6 n_none=0 | TDMI(W+M) mean=  6.7928 ( -0.1%) | null M mean= 1.9307, sd= 0.3681, z= -3.896
ridge=  1e-03 | n_ok=6 n_none=0 | TDMI(W+M) mean=  6.7564 ( -0.6%) | null M mean= 1.9769, sd= 0.5416, z= -2.734


KeyboardInterrupt: 